# 面试题：如何从零实现 RealNVP，并用可逆流做联合指标异常检测？

## 面试回答主线

Normalizing Flow 用一串可逆变换把复杂数据 $x$ 映射到简单基分布 $z$，通过换元公式精确计算 `log p(x)=log p(z)+log|det J|`。RealNVP 的 affine coupling 固定一部分维度，用它预测另一部分的 scale 与 shift，因此 Jacobian 为三角矩阵，log-det 只需累加 scale。相邻层交替 mask 后，两维都能被变换；采样则必须按相反层序逐层求逆。相比重建误差，flow 直接给出联合密度，但异常阈值仍需在独立正常验证集上校准。下面不用现成 flow 库，只用 `nn.Parameter`、矩阵运算和 autograd 手写 coupling、似然、逆变换和训练。

## 真实案例：API 延迟与错误率的联合异常

某接口存在一条正常的非线性运行带：负载增大时延迟变化，错误率沿曲线缓慢变化。单看延迟或错误率的边际都可能正常，但二者组合偏离运行带就是异常。我们用 600 个离线正常窗口训练，并准备 6 个正常、6 个“边际正常但组合异常”的可读监控窗口；数值是教学构造，不是线上 SLO 数据。

In [1]:
import math  # 导入圆周率和对数常数以计算标准高斯密度。
import torch  # 导入 PyTorch 以手写可逆流并真实反向传播。
from torch import nn  # 导入基础模块和可学习参数。
torch.set_num_threads(1)  # 固定二维教学实验为单线程。
torch.manual_seed(83)  # 固定正常数据、参数初始化与训练输出。
def generate_normal_windows(count, seed):  # 从弯曲运行带生成确定性正常指标窗口。
    generator = torch.Generator().manual_seed(seed)  # 为当前数据集创建独立随机源。
    latent_load = torch.randn(count, generator=generator)  # 采样标准化负载因素。
    sensor_noise = torch.randn(count, generator=generator)  # 采样错误率方向的小幅噪声。
    latency_axis = latent_load * 1.05  # 把负载映射到标准化延迟轴。
    error_axis = 0.48 * latency_axis.pow(2) - 0.45 + sensor_noise * 0.16  # 构造带厚度的抛物线正常流形。
    return torch.stack([latency_axis, error_axis], dim=-1)  # 返回二维标准化监控特征。
train_windows = generate_normal_windows(600, 300)  # 创建只含正常窗口的训练集。
validation_windows = generate_normal_windows(160, 301)  # 创建独立正常阈值校准集。
normal_test = torch.tensor([[-1.50, 0.68], [-0.90, -0.10], [-0.25, -0.42], [0.35, -0.36], [0.95, 0.02], [1.45, 0.62]])  # 定义六个沿正常曲线的测试窗口。
anomaly_test = torch.tensor([[-1.50, -0.42], [-1.05, 0.72], [-0.25, 0.62], [0.35, 0.72], [1.00, -0.48], [1.50, -0.38]])  # 定义六个边际合理但偏离联合曲线的异常窗口。
test_windows = torch.cat([normal_test, anomaly_test], dim=0)  # 合并正常与异常测试窗口。
test_targets = torch.tensor([0] * len(normal_test) + [1] * len(anomaly_test))  # 用零表示正常、一表示异常。
window_names = [f"N{index + 1}" for index in range(len(normal_test))] + [f"A{index + 1}" for index in range(len(anomaly_test))]  # 创建可追踪窗口编号。
def to_business_metrics(standardized):  # 把标准化特征还原成可读监控单位。
    latency_ms = 180.0 + standardized[..., 0] * 35.0  # 将第一维映射为 P95 延迟毫秒。
    error_percent = 2.5 + standardized[..., 1] * 1.6  # 将第二维映射为错误率百分比。
    return latency_ms, error_percent  # 返回业务单位指标。
test_latency, test_error = to_business_metrics(test_windows)  # 还原十二个测试窗口指标。
print("窗口  gold  P95延迟(ms)  错误率(%)  标准化坐标")  # 输出真实案例预览表头。
for index, name in enumerate(window_names):  # 遍历全部正常与异常窗口。
    gold = "异常" if int(test_targets[index]) else "正常"  # 还原人工标签名称。
    coordinates = [round(float(value), 2) for value in test_windows[index]]  # 形成可读标准化坐标。
    print(f"{name:<3}   {gold}    {float(test_latency[index]):>7.1f}       {float(test_error[index]):>5.2f}     {coordinates}")  # 展示边际值与联合异常语义。

窗口  gold  P95延迟(ms)  错误率(%)  标准化坐标
N1    正常      127.5        3.59     [-1.5, 0.68]
N2    正常      148.5        2.34     [-0.9, -0.1]
N3    正常      171.2        1.83     [-0.25, -0.42]
N4    正常      192.2        1.92     [0.35, -0.36]
N5    正常      213.2        2.53     [0.95, 0.02]
N6    正常      230.8        3.49     [1.45, 0.62]
A1    异常      127.5        1.83     [-1.5, -0.42]
A2    异常      143.2        3.65     [-1.05, 0.72]
A3    异常      171.2        3.49     [-0.25, 0.62]
A4    异常      192.2        3.65     [0.35, 0.72]
A5    异常      215.0        1.73     [1.0, -0.48]
A6    异常      232.5        1.89     [1.5, -0.38]


## Baseline（基线）：独立高斯边际密度

基线分别拟合延迟与错误率的均值、方差，再把两个一维负对数似然相加。它能发现单指标极端值，却忽略“给定延迟时错误率应该落在哪里”的非线性依赖。阈值只用独立正常验证集的 95% 分位数确定。

In [2]:
baseline_mean = train_windows.mean(dim=0)  # 估计两个边际的训练均值。
baseline_variance = train_windows.var(dim=0, unbiased=False).clamp_min(1e-6)  # 估计两个边际方差并防止除零。
def independent_gaussian_nll(windows):  # 计算忽略相关性的独立高斯负对数似然。
    standardized_square = (windows - baseline_mean).pow(2) / baseline_variance  # 计算每个边际的标准化平方偏差。
    log_variance = torch.log(2.0 * math.pi * baseline_variance)  # 计算每个边际的归一化常数。
    return 0.5 * (standardized_square + log_variance).sum(dim=-1)  # 汇总两维得到窗口异常分数。
baseline_validation_scores = independent_gaussian_nll(validation_windows)  # 在独立正常集上计算校准分数。
baseline_threshold = float(torch.quantile(baseline_validation_scores, 0.95))  # 取正常分数 95% 分位作为阈值。
baseline_test_scores = independent_gaussian_nll(test_windows)  # 计算十二个测试窗口基线分数。
baseline_predictions = (baseline_test_scores > baseline_threshold).long()  # 超过阈值判为异常。
def classification_metrics(predictions, targets):  # 手写异常检测 precision、recall 与 F1。
    true_positive = int(((predictions == 1) & (targets == 1)).sum())  # 统计正确检出的异常。
    false_positive = int(((predictions == 1) & (targets == 0)).sum())  # 统计误报告警。
    false_negative = int(((predictions == 0) & (targets == 1)).sum())  # 统计漏报异常。
    precision = true_positive / max(true_positive + false_positive, 1)  # 计算告警精确率。
    recall = true_positive / max(true_positive + false_negative, 1)  # 计算异常召回率。
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-9)  # 计算精确率召回率调和平均。
    return precision, recall, f1, true_positive, false_positive, false_negative  # 返回指标和混淆计数。
baseline_metrics = classification_metrics(baseline_predictions, test_targets)  # 汇总独立边际基线效果。
print(f"独立高斯阈值（正常验证95%分位）：{baseline_threshold:.3f}")  # 展示可复现阈值来源。
print("窗口 gold  独立NLL  预测  正确")  # 输出逐窗口基线表头。
for index, name in enumerate(window_names):  # 遍历十二个测试窗口。
    predicted = "异常" if int(baseline_predictions[index]) else "正常"  # 还原基线预测名称。
    gold = "异常" if int(test_targets[index]) else "正常"  # 还原人工标签名称。
    print(f"{name:<3}  {gold}   {float(baseline_test_scores[index]):>7.3f}  {predicted}  {bool(baseline_predictions[index] == test_targets[index])}")  # 展示边际模型的具体漏报。
print(f"Baseline precision={baseline_metrics[0]:.3f}，recall={baseline_metrics[1]:.3f}，F1={baseline_metrics[2]:.3f}")  # 输出基线检测指标。

独立高斯阈值（正常验证95%分位）：5.955
窗口 gold  独立NLL  预测  正确
N1   正常     3.016  正常  True
N2   正常     2.090  正常  True
N3   正常     1.901  正常  True
N4   正常     1.859  正常  True
N5   正常     2.035  正常  True
N6   正常     2.780  正常  True
A1   异常     2.935  正常  False
A2   异常     2.521  正常  False
A3   异常     1.928  正常  False
A4   异常     2.024  正常  False
A5   异常     2.324  正常  False
A6   异常     2.787  正常  False
Baseline precision=0.000，recall=0.000，F1=0.000


## 核心实现：交替 mask 的 affine coupling

每层保留 `mask==1` 的维度不变，用它经过一个小 MLP 预测另一维的 `log_scale` 与 `shift`。`tanh * 0.8` 限制早期缩放，正向 log-det 是被变换维度的 `log_scale` 之和；逆向则执行 `(y-shift)*exp(-log_scale)`。

In [3]:
class AffineCoupling(nn.Module):  # 定义二维 RealNVP 的单个仿射耦合层。
    def __init__(self, mask, hidden_dim=24):  # 初始化固定 mask 与 scale/shift 小网络。
        super().__init__()  # 注册基础模块状态。
        self.register_buffer("mask", torch.tensor(mask, dtype=torch.float32))  # 保存不参与训练的维度选择 mask。
        self.hidden_weight = nn.Parameter(torch.randn(2, hidden_dim) * 0.16)  # 创建被保留维到隐藏层的权重。
        self.hidden_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建隐藏层偏置。
        self.scale_weight = nn.Parameter(torch.randn(hidden_dim, 2) * 0.08)  # 创建 log-scale 输出权重。
        self.scale_bias = nn.Parameter(torch.zeros(2))  # 创建 log-scale 偏置。
        self.shift_weight = nn.Parameter(torch.randn(hidden_dim, 2) * 0.08)  # 创建 shift 输出权重。
        self.shift_bias = nn.Parameter(torch.zeros(2))  # 创建 shift 偏置。
    def parameters_from_fixed_part(self, values):  # 用不变维度预测仿射参数。
        fixed = values * self.mask  # 清零本层要被变换的输入维度。
        hidden = torch.tanh(fixed @ self.hidden_weight + self.hidden_bias)  # 提取不变部分的非线性表示。
        log_scale = torch.tanh(hidden @ self.scale_weight + self.scale_bias) * 0.8  # 生成有界对数缩放。
        shift = hidden @ self.shift_weight + self.shift_bias  # 生成平移量。
        active = 1.0 - self.mask  # 标出本层实际变换的维度。
        return log_scale * active, shift * active  # 确保固定维的仿射参数严格为零。
    def forward(self, values, reverse=False):  # 执行正向密度变换或逆向采样变换。
        log_scale, shift = self.parameters_from_fixed_part(values)  # 根据固定维计算当前仿射参数。
        if reverse:  # 逆变换需要撤销 shift 与 scale。
            transformed = values * self.mask + (1.0 - self.mask) * (values - shift) * torch.exp(-log_scale)  # 精确恢复本层输入。
            log_determinant = -log_scale.sum(dim=-1)  # 逆向 Jacobian 对数行列式取负。
        else:  # 正向变换把数据映射到更简单空间。
            transformed = values * self.mask + (1.0 - self.mask) * (values * torch.exp(log_scale) + shift)  # 对活动维执行仿射映射。
            log_determinant = log_scale.sum(dim=-1)  # 三角 Jacobian 的 log-det 等于缩放之和。
        return transformed, log_determinant  # 返回变换值与逐样本 log-det。
class ManualRealNVP(nn.Module):  # 定义由六个交替 coupling 组成的二维可逆流。
    def __init__(self):  # 创建交替固定第一维和第二维的层序列。
        super().__init__()  # 注册基础模块状态。
        masks = ([1.0, 0.0], [0.0, 1.0]) * 3  # 让两维轮流作为条件和被变换对象。
        self.layers = nn.ModuleList([AffineCoupling(mask) for mask in masks])  # 注册六个独立仿射耦合层。
    def transform(self, values):  # 把数据 x 顺序映射到基变量 z。
        transformed = values  # 从原始二维监控值开始。
        total_log_determinant = torch.zeros(values.shape[0], device=values.device)  # 初始化逐样本 log-det 累计量。
        intermediates = [values]  # 保存各层输出以解释变换过程。
        for layer in self.layers:  # 按模型顺序执行全部 coupling。
            transformed, log_determinant = layer(transformed)  # 应用当前可逆仿射变换。
            total_log_determinant = total_log_determinant + log_determinant  # 累加换元公式所需 log-det。
            intermediates.append(transformed)  # 保存本层后的二维坐标。
        return transformed, total_log_determinant, intermediates  # 返回基变量、log-det 与路径。
    def inverse(self, latent):  # 把基变量按相反层序映射回数据空间。
        reconstructed = latent  # 从标准高斯样本开始。
        for layer in reversed(self.layers):  # 逆变换必须严格反转层顺序。
            reconstructed, _ = layer(reconstructed, reverse=True)  # 撤销当前 coupling 的仿射操作。
        return reconstructed  # 返回精确恢复或生成的数据样本。
    def log_prob(self, values):  # 用换元公式计算数据精确对数密度。
        latent, log_determinant, _ = self.transform(values)  # 把数据映射到标准高斯空间。
        base_log_prob = -0.5 * (latent.pow(2) + math.log(2.0 * math.pi)).sum(dim=-1)  # 计算二维标准高斯 log p(z)。
        return base_log_prob + log_determinant  # 加上 Jacobian 修正得到 log p(x)。
torch.manual_seed(89)  # 固定 RealNVP 参数初始化。
flow = ManualRealNVP()  # 创建六层手写可逆流。
preview_latent, preview_log_det, preview_path = flow.transform(test_windows[:2])  # 对两个窗口观察未训练变换路径。
preview_reconstruction = flow.inverse(preview_latent)  # 使用正确逆序恢复原始输入。
print("前两窗口 x：", [[round(float(value), 3) for value in row] for row in test_windows[:2]])  # 展示变换起点。
print("六层后 z：", [[round(float(value), 3) for value in row] for row in preview_latent])  # 展示基空间坐标。
print("逐层首窗口坐标：", [[round(float(value), 3) for value in point[0]] for point in preview_path])  # 展示 coupling 如何交替修改两维。
print("log|det J|：", [round(float(value), 4) for value in preview_log_det])  # 展示密度修正项而非只给输出 shape。
print("未训练正逆最大误差：", f"{float((preview_reconstruction - test_windows[:2]).abs().max()):.8f}")  # 验证可逆结构的数值路径。

前两窗口 x： [[-1.5, 0.68], [-0.9, -0.1]]
六层后 z： [[-1.508, 0.427], [-0.895, -0.302]]
逐层首窗口坐标： [[-1.5, 0.68], [-1.5, 0.56], [-1.493, 0.56], [-1.493, 0.418], [-1.505, 0.418], [-1.505, 0.427], [-1.508, 0.427]]
log|det J|： [0.0896, 0.0539]
未训练正逆最大误差： 0.00000012


## 最大似然训练、阈值校准与结果表

训练目标是正常窗口平均负 log-likelihood。阈值仍只从独立正常验证集的 95% 分位数获得，不能偷看异常标签调参。结果同时展示正常/异常分数、precision、recall、F1、latent 坐标和模型生成样本。

In [4]:
optimizer = torch.optim.Adam(flow.parameters(), lr=0.006)  # 创建更新所有 coupling 参数的 Adam。
training_history = []  # 保存负对数似然和首层梯度轨迹。
for epoch in range(700):  # 对正常运行带执行最大似然训练。
    optimizer.zero_grad()  # 清除上一轮累计梯度。
    negative_log_likelihood = -flow.log_prob(train_windows).mean()  # 计算正常样本平均负 log p(x)。
    negative_log_likelihood.backward()  # 从精确似然反向传播到 scale 与 shift 网络。
    gradient_norm = float(flow.layers[0].scale_weight.grad.norm().detach())  # 观察首层缩放分支真实梯度。
    torch.nn.utils.clip_grad_norm_(flow.parameters(), 5.0)  # 防止偶发极端 scale 梯度。
    optimizer.step()  # 按真实梯度更新可逆流。
    training_history.append((float(negative_log_likelihood.detach()), gradient_norm))  # 保存优化过程。
with torch.no_grad():  # 关闭校准和测试阶段梯度记录。
    flow_validation_scores = -flow.log_prob(validation_windows)  # 计算独立正常集异常分数。
    flow_threshold = float(torch.quantile(flow_validation_scores, 0.95))  # 用正常 95% 分位设置 flow 阈值。
    flow_test_scores = -flow.log_prob(test_windows)  # 计算十二个窗口的联合异常分数。
    flow_latent, flow_log_det, _ = flow.transform(test_windows)  # 获取测试窗口基空间与 Jacobian 项。
    generated_windows = flow.inverse(torch.randn(5, 2))  # 从标准高斯逆变换生成五个正常候选窗口。
flow_predictions = (flow_test_scores > flow_threshold).long()  # 超过联合密度阈值判为异常。
flow_metrics = classification_metrics(flow_predictions, test_targets)  # 计算 RealNVP precision、recall 与 F1。
generated_latency, generated_error = to_business_metrics(generated_windows)  # 把生成样本还原为业务单位。
print("阶段        NLL       首层scale梯度")  # 输出最大似然训练轨迹表头。
for epoch in (0, 99, 349, 699):  # 选择四个关键训练时刻。
    print(f"{epoch + 1:>4}      {training_history[epoch][0]:>7.4f}       {training_history[epoch][1]:>8.5f}")  # 展示 loss 与梯度随训练变化。
print(f"RealNVP 正常验证95%阈值：{flow_threshold:.3f}")  # 展示联合密度阈值。
print("窗口 gold  独立NLL  Flow NLL  Flow预测  z坐标")  # 输出逐窗口对照表头。
for index, name in enumerate(window_names):  # 遍历全部十二个测试窗口。
    predicted = "异常" if int(flow_predictions[index]) else "正常"  # 还原 flow 预测名称。
    latent_text = [round(float(value), 2) for value in flow_latent[index]]  # 格式化二维基空间位置。
    print(f"{name:<3}  {'异常' if int(test_targets[index]) else '正常'}   {float(baseline_test_scores[index]):>7.3f}   {float(flow_test_scores[index]):>7.3f}    {predicted}    {latent_text}")  # 对比边际与联合密度。
print(f"指标对比：Baseline F1={baseline_metrics[2]:.3f}，RealNVP precision={flow_metrics[0]:.3f}，recall={flow_metrics[1]:.3f}，F1={flow_metrics[2]:.3f}")  # 汇总同一测试集效果。
print("Flow 生成的五个候选正常窗口（延迟ms, 错误率%）：")  # 输出采样结果标题。
for latency, error in zip(generated_latency, generated_error):  # 遍历五个生成监控点。
    print(f"({float(latency):.1f}, {float(error):.2f})")  # 展示逆变换不只是理论公式。

阶段        NLL       首层scale梯度
   1       2.7414        0.36481
 100       1.1173        0.02765
 350       1.0472        0.02777
 700       1.0335        0.01064
RealNVP 正常验证95%阈值：3.295
窗口 gold  独立NLL  Flow NLL  Flow预测  z坐标
N1   正常     3.016     1.030    正常    [-1.26, 0.81]
N2   正常     2.090     0.390    正常    [-1.05, -0.3]
N3   正常     1.901    -0.003    正常    [-0.34, -0.27]
N4   正常     1.859     0.109    正常    [0.26, -0.01]
N5   正常     2.035     0.539    正常    [0.81, 0.03]
N6   正常     2.780     1.128    正常    [1.2, 0.32]
A1   异常     2.935    10.895    异常    [-1.22, -4.44]
A2   异常     2.521    15.798    异常    [-1.41, 5.56]
A3   异常     1.928    44.479    异常    [-1.29, 9.45]
A4   异常     2.024    49.104    异常    [-0.54, 9.99]
A5   异常     2.324     4.447    异常    [1.48, -2.51]
A6   异常     2.787    11.143    异常    [2.6, -3.83]
指标对比：Baseline F1=0.000，RealNVP precision=1.000，recall=1.000，F1=1.000
Flow 生成的五个候选正常窗口（延迟ms, 错误率%）：
(168.7, 2.01)
(156.3, 2.11)
(157.5, 2.41)
(159.3, 2.39)
(198.0, 1.7

## 结果解读

独立高斯只知道两维各自常见范围，因此会漏掉位于边际中心、却远离抛物线运行带的组合异常。RealNVP 通过交替 coupling 把弯曲正常带拉直到近似标准高斯，偏离条件关系的窗口在 latent 空间更极端，联合 NLL 更高。训练 loss、scale 梯度、逐窗口分数和生成样本共同证明这里执行了真实密度建模；小型合成流形上的 F1 不能外推到线上。

## 失败案例：逆变换仍按正向层序执行

复合函数的逆必须反转调用顺序。下面故意用正向顺序逐层 inverse；每个 coupling 单独都可逆，但组合后无法回到原点。正确 `flow.inverse` 使用 `reversed(self.layers)`。

In [5]:
with torch.no_grad():  # 关闭故障复现的梯度记录。
    probe_input = test_windows[:6]  # 选择六个真实测试窗口作为可逆性探针。
    probe_latent, _, _ = flow.transform(probe_input)  # 用训练后模型执行正确正向变换。
    wrong_reconstruction = probe_latent  # 从相同 latent 开始错误逆变换。
    for layer in flow.layers:  # 故意保持正向层序而不是反转。
        wrong_reconstruction, _ = layer(wrong_reconstruction, reverse=True)  # 每层公式正确但组合顺序错误。
    correct_reconstruction = flow.inverse(probe_latent)  # 用反转层序执行正确复原。
wrong_error = float((wrong_reconstruction - probe_input).abs().max())  # 计算错误层序最大重建偏差。
correct_error = float((correct_reconstruction - probe_input).abs().max())  # 计算正确逆变换数值误差。
print("窗口  原始x            错序逆变换         正确逆变换")  # 输出逐窗口可逆性对照表头。
for index in range(len(probe_input)):  # 遍历六个探针窗口。
    original = [round(float(value), 3) for value in probe_input[index]]  # 格式化原始二维坐标。
    wrong = [round(float(value), 3) for value in wrong_reconstruction[index]]  # 格式化错误恢复结果。
    correct = [round(float(value), 3) for value in correct_reconstruction[index]]  # 格式化正确恢复结果。
    print(f"{window_names[index]:<3}   {original}   {wrong}   {correct}")  # 展示层序错误造成的实际漂移。
print(f"最大重建误差：错误层序={wrong_error:.6f}，反向层序={correct_error:.8f}")  # 量化修复收益。

窗口  原始x            错序逆变换         正确逆变换
N1    [-1.5, 0.68]   [-0.242, -0.387]   [-1.5, 0.68]
N2    [-0.9, -0.1]   [-0.124, -0.689]   [-0.9, -0.1]
N3    [-0.25, -0.42]   [0.015, -0.393]   [-0.25, -0.42]
N4    [0.35, -0.36]   [0.085, 0.101]   [0.35, -0.36]
N5    [0.95, 0.02]   [0.172, 0.524]   [0.95, 0.02]
N6    [1.45, 0.62]   [0.231, 0.954]   [1.45, 0.62]
最大重建误差：错误层序=1.258477，反向层序=0.00000036


## 生产差距与追问

真实监控需要按服务版本和时段切分、处理季节性与概念漂移、缺失值、单位变更、阈值告警预算和根因解释。高 likelihood 不等于语义正常，flow 可能对 OOD 数据给出反直觉密度；应结合规则、条件模型、conformal 校准和人工复核。高维场景还要监控 log-det 数值范围、可逆层版本与逆向采样稳定性。

## 最小回归测试

In [6]:
assert len(test_windows) >= 5  # 保证异常检测案例数量足以形成逐样本对照。
assert flow_metrics[2] > baseline_metrics[2]  # 保护联合密度模型确实改善独立边际基线。
assert correct_error < 1e-5  # 保护正确逆层序保持数值可逆。
assert wrong_error > correct_error * 100.0  # 保护失败案例确实暴露组合层序错误。
assert torch.isfinite(flow_test_scores).all()  # 保护全部测试窗口得到有限精确似然。
print("最小回归测试通过：联合密度、阈值评估和逆层序修复均保持有效。")  # 输出集中测试结论。

最小回归测试通过：联合密度、阈值评估和逆层序修复均保持有效。
